# NB_T1b · Multi-agents LangGraph — GR509 Crêt de la Neige

🏗️ Build | 🚢 Ship | 📤 Share · ⏱️ ~90 min

### 🏗️ Build
- Partir de `create_react_agent` (5 lignes) vers un `StateGraph` explicite
- Implémenter WeatherAgent et HikingAgent comme noeuds LangGraph avec `ToolNode`
- Construire un superviseur LLM qui route dynamiquement entre les agents

### 🚢 Ship
Pipeline complet GR509 identique à NB_T1a — même résultat, autre moteur :
météo → métriques → recommandation finale. Avec persistance `MemorySaver` et HITL.

### 📤 Share
Grille Mistral Agents API vs LangGraph : quand choisir quoi, sans dogme.

**Pré-requis.**
`pip install langchain-mistralai langgraph` + `sentier_gr509.py` (même dossier).
Variable `MISTRAL_API_KEY` requise.

> **Relation avec NB_T1a.** Ce notebook implémente le **même pipeline GR509**
> que NB_T1a (Mistral Agents API) avec LangGraph. Même use case, même résultat attendu.
> L'objectif est de comparer les deux approches côte à côte.

> **Note honnête.** Le fichier GPX (`atelier/trace_gps.gpx`) peut être absent —
> `sentier_gr509.py` bascule automatiquement sur des valeurs calibrées (8,4 km / 680 m D+).


## Philosophie LangGraph vs Mistral Agents API

| Dimension | Mistral Agents API (NB_T1a) | LangGraph (ce notebook) |
|---|---|---|
| **Création agent** | `agents.create(...)` → `agent_id` persistant | Noeud dans un `StateGraph` |
| **Dispatch tools** | Boucle `function.call` + `FunctionResultEntry` | `ToolNode` automatique |
| **Handoff** | Natif plateforme (`handoffs=[id]`) | `add_conditional_edges` + superviseur |
| **État** | Côté serveur Mistral (`conversation_id`) | Côté client (dict + `MemorySaver`) |
| **HITL** | Via Workflows `wait_for_input` | Via `interrupt` + `Command(resume=…)` |
| **Portabilité** | Lié à Mistral | Open-source, multi-provider |

> **Résumé.** Mistral Agents API = démarrage rapide, état géré côté serveur.
> LangGraph = ownership de l'état côté client, portable, HITL avancé.
> Ni l'un ni l'autre n'est universellement meilleur — contexte décide.


---

## Task 1 · Premier contact : create_react_agent (prébâti)


**La voie la plus rapide.** `create_react_agent` de LangGraph crée un agent
ReAct complet en 3 lignes : `llm`, `tools`, `invoke`. Pas de `StateGraph` explicite.
C'est l'équivalent direct de `agents.create` + `conversations.start` en Mistral.

Différence clé : l'état (messages) reste côté client dans un dict Python.

```
Mistral Agents API          LangGraph
───────────────────         ──────────────────────────────
agents.create(...)          create_react_agent(llm, tools)
conversations.start(...)    agent.invoke({'messages': [...]})
État : côté serveur         État : dict Python côté client
```

> **À retenir.** `create_react_agent` est le point d'entrée recommandé.
> Passer à `StateGraph` explicite seulement si besoin de noeuds multiples ou de routage.


In [1]:
import os, json
from pathlib import Path
from langchain_mistralai import ChatMistralAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from sentier_gr509 import (
    ITINERAIRE_GR509, charger_trace_gpx,
    calculer_distance_km, denivele_positif_m,
    estimation_naismith, get_meteo_prevision,
)

MODELE = os.environ.get('MISTRAL_MODEL', 'mistral-small-latest')
_cle   = os.environ.get('MISTRAL_API_KEY')
_url   = os.environ.get('MISTRAL_SERVER_URL')
# Gotcha serveur dédié : langchain-mistralai poste sur {base}/chat/completions → ajouter /v1
_ep  = (_url.rstrip('/') + '/v1') if _url else None
llm  = (ChatMistralAI(model=MODELE, api_key=_cle, endpoint=_ep)
        if _ep else ChatMistralAI(model=MODELE, api_key=_cle))

it      = ITINERAIRE_GR509
gpx     = charger_trace_gpx(it.fichier_gpx)
DIST_KM = calculer_distance_km(gpx)
DENIV_M = denivele_positif_m(gpx)

print('╔═══════════════════════════════════════════════════════╗')
print('║  🦜🔗  LangGraph · Environnement prêt                ║')
print('╚═══════════════════════════════════════════════════════╝')
print(f'  🤖 Modèle     : {MODELE}')
print(f'  🗺️  Itinéraire  : {it.nom}')
print(f'  📏 Distance    : {DIST_KM} km')
print(f'  ⛰️  Dénivelé D+ : {DENIV_M} m')
print(f'  ⏱️  Naismith    : {estimation_naismith(DIST_KM, DENIV_M)} h')
print(f'  📡 Source GPS  : {"GPX réel" if gpx else "fallback calibré (8,4 km / 680 m)"}')


C:\Users\mickael.labarrere\OneDrive - Accenture\Formation Mistral\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


╔═══════════════════════════════════════════════════════╗
║  🦜🔗  LangGraph · Environnement prêt                ║
╚═══════════════════════════════════════════════════════╝
  🤖 Modèle     : mistral-small-latest
  🗺️  Itinéraire  : Col de Menthières → Crêt de la Neige
  📏 Distance    : 8.4 km
  ⛰️  Dénivelé D+ : 680 m
  ⏱️  Naismith    : 2.81 h
  📡 Source GPS  : fallback calibré (8,4 km / 680 m)


C:\Users\mickael.labarrere\OneDrive - Accenture\Formation Mistral\.venv\Lib\site-packages\langgraph\checkpoint\base\__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
# ── Définir les tools avec @tool ──────────────────────────────────────────────
@tool
def tool_meteo(lieu: str) -> dict:
    '''Retourne la météo actuelle pour un lieu. Ne jamais inventer une valeur météo.'''
    return get_meteo_prevision(lieu)

@tool
def tool_distance() -> dict:
    '''Calcule la distance totale du GR509 en km.'''
    return {'distance_km': calculer_distance_km(gpx)}

@tool
def tool_denivele() -> dict:
    '''Retourne le dénivelé positif cumulé en mètres.'''
    return {'denivele_m': denivele_positif_m(gpx)}

@tool
def tool_naismith(dist_km: float, deniv_pos_m: int) -> dict:
    '''Estime la durée de marche (règle Naismith : dist_km/5 + deniv_pos_m/600).'''
    return {'duree_h': estimation_naismith(dist_km, deniv_pos_m)}

# ── WeatherAgent avec create_react_agent ─────────────────────────────────────
_sys = 'Tu es un agent météo. RÈGLE : appelle tool_meteo pour toute question météo.'
weather_react = create_react_agent(llm.bind(system=_sys), tools=[tool_meteo])
result = weather_react.invoke({'messages': [('user', f'Météo au {it.arrivee} ?')]})
rep = result['messages'][-1].content

print('╔═══════════════════════════════════════════════════════╗')
print('║  🌤️  WeatherAgent — create_react_agent               ║')
print('╚═══════════════════════════════════════════════════════╝')
print(f'  📍 Lieu    : {it.arrivee}')
print(f'  🤖 Réponse : {rep}')


╔═══════════════════════════════════════════════════════╗
║  🌤️  WeatherAgent — create_react_agent               ║
╚═══════════════════════════════════════════════════════╝
  📍 Lieu    : Crêt de la Neige
  🤖 Réponse : La météo au Crêt de la Neige est actuellement partiellement nuageuse avec une température de 8.0°C et des vents soufflant à 25.0 km/h.


---

## Task 2 · StateGraph explicite + MessagesState


**Pourquoi aller plus loin que `create_react_agent` ?**
Quand on veut plusieurs noeuds (WeatherAgent ET HikingAgent), chacun avec
ses propres tools et son propre système prompt, il faut un `StateGraph` explicite.

**Les 3 briques.**

| Brique | Rôle | Équivalent Mistral |
|---|---|---|
| `MessagesState` | État partagé entre noeuds (liste de messages) | `conversation_id` côté serveur |
| `add_node` | Enregistre un noeud agent ou tool | Crée un agent persistant |
| `compile()` | Fige le graphe pour exécution | — (implicite dans Agents API) |

> **Ownership de l'état.** En LangGraph, `MessagesState` est un dict Python
> dans votre process. En Mistral Agents API, l'état vit côté serveur Mistral
> (accessible seulement via `conversation_id`).


In [3]:
from langgraph.graph import StateGraph, END
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import HumanMessage

_llm_weather = llm.bind_tools([tool_meteo])
def weather_node(state: MessagesState):
    msgs = [('system', 'Tu es un agent météo expert du Jura.\n'
                        'RÈGLE ABSOLUE : pour toute question météo, appelle tool_meteo.')
            ] + state['messages']
    return {'messages': [_llm_weather.invoke(msgs)]}

g = StateGraph(MessagesState)
g.add_node('weather_agent', weather_node)
g.add_node('weather_tools', ToolNode([tool_meteo]))
g.set_entry_point('weather_agent')
g.add_conditional_edges('weather_agent', tools_condition, {'tools': 'weather_tools', END: END})
g.add_edge('weather_tools', 'weather_agent')
weather_graph = g.compile()

result = weather_graph.invoke({'messages': [HumanMessage(f'Météo au {it.arrivee} ?')]})
rep = result['messages'][-1].content
n_msgs = len(result['messages'])

print('╔═══════════════════════════════════════════════════════╗')
print('║  🗺️  WeatherAgent · StateGraph explicite             ║')
print('╚═══════════════════════════════════════════════════════╝')
print(f'  📊 Messages dans l\'état : {n_msgs}')
print(f'  📍 Lieu     : {it.arrivee}')
print(f'  🤖 Réponse  : {rep}')
print()
print('  Flux d\'exécution :')
print('  weather_agent → (tool_call) → weather_tools → weather_agent → END')


╔═══════════════════════════════════════════════════════╗
║  🗺️  WeatherAgent · StateGraph explicite             ║
╚═══════════════════════════════════════════════════════╝
  📊 Messages dans l'état : 4
  📍 Lieu     : Crêt de la Neige
  🤖 Réponse  : La météo au Crêt de la Neige est la suivante :
- Température : 13.5 °C
- Vent : 14.0 km/h
- Condition : Données météo réelles Open-Meteo

Source : open-meteo

  Flux d'exécution :
  weather_agent → (tool_call) → weather_tools → weather_agent → END


---

## Task 3 · HikingAgent : ToolNode automatique


**`ToolNode` vs dispatch manuel.** En Mistral Agents API (NB_T1a),
on écrit une boucle `function.call → execute_tool → FunctionResultEntry` à la main.
En LangGraph, `ToolNode([tool_distance, tool_denivele, tool_naismith])` automatise
exactement cette boucle — même comportement, moins de code.

```
Mistral Agents API (NB_T1a)         LangGraph (ce notebook)
──────────────────────────           ───────────────────────
for call in outputs:                 ToolNode([tool1, tool2])
    result = execute_tool(call)      # 1 ligne, dispatch automatique
    FunctionResultEntry(...)         # routing conditionnel géré par
conversations.append(results)       # add_conditional_edges
```

> **Piège.** `ToolNode` appelle les tools de façon synchrone par défaut.
> Pour des tools avec I/O réseau lentes, utiliser `ToolNode` en mode async.


In [4]:
def un_seul_appel(reponse):
    """Ne garde que le premier tool_call : la boucle devient strictement séquentielle.

    Le modèle a tendance à demander distance ET dénivelé d'un coup. L'ordre imposé
    dans le prompt système n'est alors plus respecté, et un serveur qui ne gère pas
    les appels parallèles rejette la requête suivante. Tronquer force un aller-retour
    par outil : c'est plus lent, mais l'ordre est garanti et la trace est lisible.
    """
    tc = getattr(reponse, 'tool_calls', None)
    return reponse.model_copy(update={'tool_calls': tc[:1]}) if tc and len(tc) > 1 else reponse

_llm_hiking = llm.bind_tools([tool_distance, tool_denivele, tool_naismith])
def hiking_node(state: MessagesState):
    msgs = [('system', 'Tu analyses les métriques physiques de la montée GR509.\n'
                        'ORDRE OBLIGATOIRE : 1) tool_distance, 2) tool_denivele, 3) tool_naismith.')
            ] + state['messages']
    return {'messages': [un_seul_appel(_llm_hiking.invoke(msgs))]}

g2 = StateGraph(MessagesState)
g2.add_node('hiking_agent', hiking_node)
g2.add_node('hiking_tools', ToolNode([tool_distance, tool_denivele, tool_naismith]))
g2.set_entry_point('hiking_agent')
g2.add_conditional_edges('hiking_agent', tools_condition, {'tools': 'hiking_tools', END: END})
g2.add_edge('hiking_tools', 'hiking_agent')
hiking_graph = g2.compile()

result_h = hiking_graph.invoke(
    {'messages': [HumanMessage(f'Métriques GR509 {it.depart} → {it.arrivee}.')]}
)
rep_h = result_h['messages'][-1].content
tool_calls_h = [tc for m in result_h['messages']
                if hasattr(m, 'tool_calls') for tc in (m.tool_calls or [])]

print('╔═══════════════════════════════════════════════════════╗')
print('║  ⛰️  HikingAgent · ToolNode automatique              ║')
print('╚═══════════════════════════════════════════════════════╝')
print(f'  🔧 Tools appelés   : {" → ".join(t["name"] for t in tool_calls_h)}')
print(f'  📏 Distance        : {DIST_KM} km')
print(f'  ⛰️  Dénivelé D+    : {DENIV_M} m')
print(f'  ⏱️  Naismith        : {estimation_naismith(DIST_KM, DENIV_M)} h')
print()
print(f'  🤖 Réponse agent : {rep_h}')


╔═══════════════════════════════════════════════════════╗
║  ⛰️  HikingAgent · ToolNode automatique              ║
╚═══════════════════════════════════════════════════════╝
  🔧 Tools appelés   : tool_distance → tool_denivele → tool_naismith
  📏 Distance        : 8.4 km
  ⛰️  Dénivelé D+    : 680 m
  ⏱️  Naismith        : 2.81 h

  🤖 Réponse agent : Voici les métriques physiques de la montée GR509 du Col de Menthières au Crêt de la Neige :

1. **Distance :** 8.4 km
2. **Dénivelé positif cumulé :** 680 m
3. **Durée estimée de marche (règle de Naismith) :** 2.81 heures


---

## Task 4 · Handoff : routage conditionnel vs handoff natif


**Pas de `handoffs=[]` en LangGraph.** L'Agents API Mistral (NB_T1a) offre
`handoffs=[agent_id]` : le cloud Mistral gère le transfert de contrôle.
En LangGraph, on route manuellement avec `add_conditional_edges` et un noeud
superviseur qui inspecte le dernier message pour décider quel noeud appeler.

**Pattern superviseur LangGraph :**

```python
def router(state):
    last = state['messages'][-1].content
    if 'météo' in last.lower():    return 'weather_agent'
    if 'distance' in last.lower(): return 'hiking_agent'
    return END

g.add_conditional_edges('supervisor', router,
    {'weather_agent': 'weather_agent', 'hiking_agent': 'hiking_agent', END: END})
```

> **Avantage LangGraph.** Le routage est du code Python pur — lisible, testable,
> loggable. Pas de boîte noire côté serveur.
> **Avantage Mistral.** `handoffs=[id]` en une ligne, géré par la plateforme.


In [5]:
from langchain_core.messages import SystemMessage

SUPERVISOR_SYSTEM = (
    'Tu es un orchestrateur de randonnée GR509.\n'
    'Si la question porte sur la MÉTÉO → réponds "→weather".\n'
    'Si la question porte sur MÉTRIQUES (distance, durée, D+) → réponds "→hiking".\n'
    'Sinon, réponds directement en français.\n'
    'Un seul mot de routage, rien d\'autre.'
)

# Le jeton de routage est une DÉCISION, pas un tour de conversation : il vit
# dans l'état, jamais dans `messages`. Sinon le worker rappelle le LLM avec un
# dernier message d'assistant, que l'API refuse (invalid_request_message_order).
class GR509State(MessagesState):
    route: str

def supervisor_node(state: GR509State):
    msgs = [SystemMessage(SUPERVISOR_SYSTEM)] + state['messages']
    return {'route': llm.invoke(msgs).content}

def supervisor_router(state: GR509State):
    route = (state.get('route') or '').strip()
    if '→weather' in route: return 'weather_agent'
    if '→hiking'  in route: return 'hiking_agent'
    return END

print('╔═══════════════════════════════════════════════════════╗')
print('║  🧭  Superviseur · Test de routage (2 questions)     ║')
print('╚═══════════════════════════════════════════════════════╝')
for q, expected in [
    ('Quelle météo au Crêt de la Neige ?', '→weather'),
    ('Quelle distance pour le GR509 ?', '→hiking'),
]:
    r = supervisor_node({'messages': [HumanMessage(q)]})['route']
    ok = expected in r
    icon = '✅' if ok else '❌'
    print(f'  {icon} | routage: {r:<12} | {q[:55]}')


╔═══════════════════════════════════════════════════════╗
║  🧭  Superviseur · Test de routage (2 questions)     ║
╚═══════════════════════════════════════════════════════╝
  ✅ | routage: →weather     | Quelle météo au Crêt de la Neige ?


  ✅ | routage: →hiking      | Quelle distance pour le GR509 ?


---

## Task 5 · Persistance + HITL (MemorySaver + interrupt)


**`MemorySaver`** est le checkpointer en mémoire de LangGraph.
Il persiste l'état complet du graphe entre les invocations, identifiées
par un `thread_id`. Équivalent du `conversation_id` côté serveur Mistral.

**HITL (Human-in-the-Loop) avec `interrupt`.** LangGraph permet d'interrompre
le graphe avant un noeud critique et d'attendre une validation humaine.
La reprise se fait avec `Command(resume=valeur)`.

```python
graph = builder.compile(checkpointer=MemorySaver(),
                         interrupt_before=['recommendation_node'])
graph.invoke(Command(resume={'validated': True}), config)
```

> **Contraste NB_T1a.** Mistral Agents API n'a pas de HITL natif.
> Il faut Mistral Workflows (`wait_for_input`) ou gérer l'interruption
> côté client dans la boucle tool-calling.


In [6]:
from langgraph.checkpoint.memory import MemorySaver

g3 = StateGraph(GR509State)
g3.add_node('supervisor',    supervisor_node)
g3.add_node('weather_agent', weather_node)
g3.add_node('weather_tools', ToolNode([tool_meteo]))
g3.add_node('hiking_agent',  hiking_node)
g3.add_node('hiking_tools',  ToolNode([tool_distance, tool_denivele, tool_naismith]))
g3.set_entry_point('supervisor')
g3.add_conditional_edges('supervisor', supervisor_router)
g3.add_conditional_edges('weather_agent', tools_condition, {'tools': 'weather_tools', END: END})
g3.add_conditional_edges('hiking_agent',  tools_condition, {'tools': 'hiking_tools', END: END})
g3.add_edge('weather_tools', 'weather_agent')
g3.add_edge('hiking_tools',  'hiking_agent')
full_graph = g3.compile(checkpointer=MemorySaver())
config = {'configurable': {'thread_id': 'gr509-session-1'}}

res = full_graph.invoke(
    {'messages': [HumanMessage(f'Météo au {it.arrivee} pour la randonnée ?')]},
    config,
)
res2 = full_graph.invoke(
    {'messages': [HumanMessage('Et les métriques GPS du GR509 ?')]},
    config,
)

print('╔═══════════════════════════════════════════════════════╗')
print('║  🧠 MemorySaver · Graphe complet multi-tour          ║')
print('╚═══════════════════════════════════════════════════════╝')
print(f'  🆔 thread_id  : gr509-session-1')
print(f'  📨 Tour 1 (météo)     : {res["messages"][-1].content}')
print()
print(f'  📨 Tour 2 (métriques) : {res2["messages"][-1].content}')
print()
print('  ✅ Les deux tours partagent le même état (MemorySaver).')


╔═══════════════════════════════════════════════════════╗
║  🧠 MemorySaver · Graphe complet multi-tour          ║
╚═══════════════════════════════════════════════════════╝
  🆔 thread_id  : gr509-session-1
  📨 Tour 1 (météo)     : Pour votre randonnée au Crêt de la Neige, voici les conditions météo actuelles :

- **Température :** 13.5°C
- **Vent :** 14 km/h
- **Conditions :** Données météo réelles Open-Meteo

Bonne randonnée !

  📨 Tour 2 (métriques) : Voici les métriques physiques de la montée au Crêt de la Neige via le GR509 :

- **Distance :** 8.4 km
- **Dénivelé positif :** 680 m
- **Durée estimée (règle de Naismith) :** environ 2.81 heures

  ✅ Les deux tours partagent le même état (MemorySaver).


---

## Task 6 · Superviseur LLM — routage dynamique


**Superviseur LLM vs routage par règles.** La Task 4 utilisait un routage
par mots-clés (déterministe). Un superviseur LLM route en fonction du sens
de la requête — plus robuste aux formulations variées.

| Pattern | Routage | Cas d'usage |
|---|---|---|
| Superviseur règles (Task 4) | Mots-clés Python | Questions prévisibles, déterminisme requis |
| Superviseur LLM (cette task) | LLM choisit | Questions variées, NLP nécessaire |

> **Note.** Le superviseur LLM consomme des tokens supplémentaires à chaque routage.
> Pour un pipeline haute fréquence, envisager un routage par règles ou embeddings.


In [7]:
QUESTIONS_GR509 = [
    f'Quelle est la météo prévue au {it.arrivee} ce week-end ?',
    f'En combien de temps peut-on monter depuis {it.depart} ?',
    f'Le vent est-il compatible avec la randonnée aujourd\'hui ?',
    f'Quelle distance sépare {it.depart} du {it.arrivee} ?',
]

print('╔══════════════════════════════════════════════════════════════════╗')
print('║  🧭  ÉVALUATION · Routage superviseur LLM                       ║')
print('╠══════════════════════════════════════════════════════════════════╣')
print(f'  {"Question":<55} {"Agent":<18}')
print(f'  {"─"*55} {"─"*18}')
for q in QUESTIONS_GR509:
    route_msg = supervisor_node({'messages': [HumanMessage(q)]})['route'].strip()
    agent = supervisor_router({'messages': [HumanMessage(q)], 'route': route_msg})
    icon = '🌤️ ' if 'weather' in agent else '⛰️ ' if 'hiking' in agent else '🔚'
    print(f'  {q[:55]:<55} {icon}{agent}')
print('╚══════════════════════════════════════════════════════════════════╝')
print()
print('  ℹ️  Routage via LLM (mistral-small) — comprend les formulations variées.')
print('  💡 Superviseur règles (Task 4) pour déterminisme strict.')


╔══════════════════════════════════════════════════════════════════╗
║  🧭  ÉVALUATION · Routage superviseur LLM                       ║
╠══════════════════════════════════════════════════════════════════╣
  Question                                                Agent             
  ─────────────────────────────────────────────────────── ──────────────────
  Quelle est la météo prévue au Crêt de la Neige ce week- 🌤️ weather_agent


  En combien de temps peut-on monter depuis Col de Menthi ⛰️ hiking_agent


  Le vent est-il compatible avec la randonnée aujourd'hui 🌤️ weather_agent
  Quelle distance sépare Col de Menthières du Crêt de la  ⛰️ hiking_agent
╚══════════════════════════════════════════════════════════════════╝

  ℹ️  Routage via LLM (mistral-small) — comprend les formulations variées.
  💡 Superviseur règles (Task 4) pour déterminisme strict.


---

## Task 7 · Métriques + trajectoire via état LangGraph


**Inspecter l'état du graphe.** LangGraph expose `get_state_history(config)`
pour reconstituer la trajectoire complète d'une session multi-agents.
C'est l'équivalent de la `tool_trace` construite manuellement en NB_T1a.

Avantage : pas besoin de maintenir une `tool_trace` séparée — LangGraph
conserve tous les checkpoints automatiquement (si `MemorySaver` est actif).

> **Piège.** `get_state_history` retourne les checkpoints dans l'ordre inverse
> (du plus récent au plus ancien). Inverser si besoin d'affichage chronologique.


In [8]:
config2 = {'configurable': {'thread_id': 'gr509-full-pipeline'}}
r1 = full_graph.invoke({'messages': [HumanMessage(f'Météo au {it.arrivee}.')]}, config2)
r2 = full_graph.invoke({'messages': [HumanMessage(f'Métriques GR509 {it.depart}→{it.arrivee}.')]}, config2)

tool_calls = [tc for m in r2['messages'] if hasattr(m, 'tool_calls') for tc in (m.tool_calls or [])]

print('╔══════════════════════════════════════════════════════════════════╗')
print('║  📊  ÉVALUATION · Pipeline complet GR509 (LangGraph)            ║')
print('╠══════════════════════════════════════════════════════════════════╣')
print(f'  📬 Messages totaux  : {len(r2["messages"])}')
print(f'  🔧 Tool calls total : {len(tool_calls)}')
print(f'  📋 Tools exécutés  : {" → ".join(t["name"] for t in tool_calls)}')
print('╠══════════════════════════════════════════════════════════════════╣')
print('  🌤️  Tour 1 — Météo')
print(f'  {r1["messages"][-1].content}')
print()
print('  ⛰️  Tour 2 — Métriques')
print(f'  {r2["messages"][-1].content}')
print('╠══════════════════════════════════════════════════════════════════╣')
print('  📐 Référence Naismith :')
print(f'     📏 {DIST_KM} km  |  ⛰️  D+ {DENIV_M} m  |  ⏱️  {estimation_naismith(DIST_KM, DENIV_M)} h')
print('╚══════════════════════════════════════════════════════════════════╝')


╔══════════════════════════════════════════════════════════════════╗
║  📊  ÉVALUATION · Pipeline complet GR509 (LangGraph)            ║
╠══════════════════════════════════════════════════════════════════╣
  📬 Messages totaux  : 12
  🔧 Tool calls total : 4
  📋 Tools exécutés  : tool_meteo → tool_distance → tool_denivele → tool_naismith
╠══════════════════════════════════════════════════════════════════╣
  🌤️  Tour 1 — Météo
  La météo au Crêt de la Neige est la suivante :

- Température : 13.5°C
- Vent : 14 km/h
- Conditions : Données météo réelles Open-Meteo

Le Crêt de la Neige est le point culminant du massif du Jura, avec une altitude de 1 720 mètres. C'est un lieu populaire pour la randonnée, le ski et d'autres activités de plein air.

  ⛰️  Tour 2 — Métriques
  Le GR509 entre le Col de Menthières et le Crêt de la Neige mesure 8.4 km avec un dénivelé positif de 680 mètres. La durée estimée de la randonnée est de 2.81 heures.
╠════════════════════════════════════════════════════════

---

## Task 8 · Grille finale Mistral vs LangGraph + exercice GR5


## Grille de décision finale

| Critère | Mistral Agents API (NB_T1a) | LangGraph (ce notebook) |
|---|---|---|
| **Setup** | 3 primitives, rapide | StateGraph + nodes + edges |
| **État** | Serveur Mistral (opaque) | Dict Python (transparent) |
| **Tools** | Schéma JSON + boucle manuelle | `@tool` + `ToolNode` auto |
| **Handoff** | Natif (`handoffs=[id]`) | `add_conditional_edges` |
| **Persistance** | `conversation_id` serveur | `MemorySaver` client |
| **HITL** | Workflows `wait_for_input` | `interrupt` + `Command` |
| **Provider** | Mistral uniquement | Tout provider LangChain |
| **Debug** | Console Mistral | `get_state_history` local |
| **Choisir si…** | Stack Mistral, itération rapide | Multi-provider, HITL, état complexe |

**Règle honnête.** Il n'y a pas de « meilleur » choix universel.
Les deux approches produisent le même résultat GR509 dans ce diptyque.
Le choix dépend de portabilité, HITL, ownership de l'état et compétences de l'équipe.


In [9]:
# ── Exercice : pipeline GR5 Pontarlier → Metabief ────────────────────────────
# TODO 1 : Dans sentier_gr509.py, ajouter ITINERAIRE_GR5
# TODO 2 : from sentier_gr509 import ITINERAIRE_GR5; it_gr5 = ITINERAIRE_GR5
# TODO 3 : Réutiliser full_graph (aucun changement de code agents)
#   config_gr5 = {'configurable': {'thread_id': 'gr5-pipeline'}}
#   full_graph.invoke({'messages': [HumanMessage(f'Métriques {it_gr5.nom}')]}, config_gr5)
# TODO 4 : Fan-out asyncio GR509 vs GR5
#   async def pipeline_async(q, tid): ...
#   gr509_r, gr5_r = await asyncio.gather(pipeline_async(...), pipeline_async(...))

print('╔══════════════════════════════════════════════════════════════════╗')
print('║  🏋️  Exercice · Étendre le pipeline au GR5                      ║')
print('╠══════════════════════════════════════════════════════════════════╣')
print(f'  📍 GR509 (référence) : {DIST_KM} km | D+ {DENIV_M} m | ⏱️  {estimation_naismith(DIST_KM, DENIV_M)} h')
print(f'  📍 GR5  (cible)      : ~12 km | D+ 650 m | ⏱️  ~3.5 h')
print('╠══════════════════════════════════════════════════════════════════╣')
print('  Décommenter les TODO ci-dessus pour lancer le pipeline GR5.')
print('  full_graph accepte n\'importe quel itinéraire sans modification ✅')
print('╚══════════════════════════════════════════════════════════════════╝')


╔══════════════════════════════════════════════════════════════════╗
║  🏋️  Exercice · Étendre le pipeline au GR5                      ║
╠══════════════════════════════════════════════════════════════════╣
  📍 GR509 (référence) : 8.4 km | D+ 680 m | ⏱️  2.81 h
  📍 GR5  (cible)      : ~12 km | D+ 650 m | ⏱️  ~3.5 h
╠══════════════════════════════════════════════════════════════════╣
  Décommenter les TODO ci-dessus pour lancer le pipeline GR5.
  full_graph accepte n'importe quel itinéraire sans modification ✅
╚══════════════════════════════════════════════════════════════════╝


---

### 📚 Pour aller plus loin

- **LangGraph StateGraph** — https://langchain-ai.github.io/langgraph/reference/graphs/
- **LangGraph create_react_agent** — https://langchain-ai.github.io/langgraph/reference/prebuilt/#create_react_agent
- **LangGraph ToolNode** — https://langchain-ai.github.io/langgraph/reference/prebuilt/#toolnode
- **LangGraph MemorySaver** — https://langchain-ai.github.io/langgraph/reference/checkpoints/#memorysaver
- **LangGraph HITL** — https://langchain-ai.github.io/langgraph/concepts/human_in_the_loop/
- **ChatMistralAI (LangChain)** — https://python.langchain.com/docs/integrations/chat/mistralai/
- **Yao et al. (2022)** « ReAct » arXiv:2210.03629 — pattern Reason+Act pour les agents LLM
- **Liu et al. (2023)** « Lost in the Middle » arXiv:2307.03172 — dégradation sur contextes longs
- **Open-Meteo API** — https://open-meteo.com/en/docs
- **Naismith's rule** — https://en.wikipedia.org/wiki/Naismith%27s_rule
- **NB_T1a_multiagents_mistral.ipynb** — même pipeline GR509 avec Mistral Agents API

> 🗂️ Références complètes de la formation :
> [`ressources/references_academiques.md`](../../ressources/references_academiques.md).
